# Árboles de decisión

## 1. Carga del Dataset

Cargamos el dataset de reservas de hotel y realizamos verificaciones iniciales.



In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import expit

# Sklearn para modelado
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Para visualización de métricas
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.calibration import calibration_curve

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración para ver más columnas
pd.set_option('display.max_columns', 50)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cargar el dataset
df = pd.read_csv("https://raw.githubusercontent.com/ami232/nebrija-mpan/main/data/hotel_bookings.csv")

print(f"Dimensiones del dataset: {df.shape}")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())

# Ver primeras filas
df.head()

In [ ]:
# Información sobre tipos de datos y valores nulos
df.info()

# Distribución de la variable objetivo
print("\nDistribución de cancelaciones:")
print(df["is_canceled"].value_counts())
print(f"\nPorcentaje de cancelaciones: {df['is_canceled'].mean()*100:.1f}%")

In [ ]:
df.describe()

## 2. Train-Test Split

Dividimos el dataset en conjuntos de entrenamiento y prueba, estratificando por la variable objetivo.

In [ ]:
# Variables iniciales para el modelo baseline
variables_modelo = [
    "lead_time",
    "adr",
    "total_of_special_requests",
    "previous_cancellations",
    "is_repeated_guest",
    "hotel",
    "deposit_type",
    "customer_type",
]

target = "is_canceled"

# Crear copia y limpiar datos básicos
df_model = df.copy()
df_model = df_model[variables_modelo + [target]].dropna()

# Crear dummies para variables categóricas
vars_categoricas = ["hotel", "deposit_type", "customer_type"]
df_dummies = pd.get_dummies(df_model, columns=vars_categoricas, drop_first=True)

# Separar features y target
X = df_dummies.drop(columns=[target])
y = df_dummies[target]
# Train-test split estratificado: 80% train, 20% test
# Estratificado: queremos la misma proporción de clases (cancelaciones/no cancelaciones) en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")
print(f"\nDistribución en train: {y_train.mean():.3f}")
print(f"Distribución en test: {y_test.mean():.3f}")

## 3. Modelo Baseline

Entrenamos un modelo de regresión logística baseline con las variables iniciales.

In [ ]:
# Entrenar modelo baseline
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

print("Modelo baseline entrenado correctamente")
print(f"\nCoeficientes del modelo:")
for feature, coef in zip(X_train.columns, baseline_model.coef_[0]):
    print(f"  {feature}: {coef:.4f}")
print(f"\nIntercepto: {baseline_model.intercept_[0]:.4f}")

## 3. Función para Métricas y Visualizaciones

Creamos una función que calcula métricas ROC AUC y muestra curvas ROC y de calibración para train y test.

In [ ]:
def calcular_metricas(model, X_train, y_train, X_test, y_test):
    """
    Calcula métricas ROC AUC y muestra curvas ROC y de calibración para conjuntos de train y test.

    Parameters:
    model: Modelo entrenado
    X_train, y_train: Datos de entrenamiento
    X_test, y_test: Datos de prueba
    """
    # Predicciones de probabilidad
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob = model.predict_proba(X_test)[:, 1]

    # Calcular AUC
    auc_train = roc_auc_score(y_train, y_train_prob)
    auc_test = roc_auc_score(y_test, y_test_prob)

    print(f"ROC AUC Train: {auc_train:.4f}")
    print(f"ROC AUC Test: {auc_test:.4f}")
    print(f"Diferencia (Train - Test): {auc_train - auc_test:.4f}")

    # Visualizar curvas ROC
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # ROC Train
    fpr_train, tpr_train, _ = roc_curve(y_train, y_train_prob)
    ax1.plot(fpr_train, tpr_train, label=f"Train AUC = {auc_train:.3f}")
    ax1.plot([0, 1], [0, 1], "k--")
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title("ROC Curve - Train")
    ax1.legend()

    # ROC Test
    fpr_test, tpr_test, _ = roc_curve(y_test, y_test_prob)
    ax2.plot(fpr_test, tpr_test, label=f"Test AUC = {auc_test:.3f}")
    ax2.plot([0, 1], [0, 1], "k--")
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curve - Test")
    ax2.legend()

    plt.tight_layout()
    plt.show()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Calibración Train
    prob_true_train, prob_pred_train = calibration_curve(
        y_train, y_train_prob, n_bins=10
    )
    ax1.plot(prob_pred_train, prob_true_train, marker="o", label="Modelo")
    ax1.plot([0, 1], [0, 1], "k--", label="Perfectamente calibrado")
    ax1.set_xlabel("Probabilidad Predicha")
    ax1.set_ylabel("Fracción Positiva Real")
    ax1.set_title("Curva de Calibración - Train")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Calibración Test
    prob_true_test, prob_pred_test = calibration_curve(y_test, y_test_prob, n_bins=10)
    ax2.plot(prob_pred_test, prob_true_test, marker="o", label="Modelo")
    ax2.plot([0, 1], [0, 1], "k--", label="Perfectamente calibrado")
    ax2.set_xlabel("Probabilidad Predicha")
    ax2.set_ylabel("Fracción Positiva Real")
    ax2.set_title("Curva de Calibración - Test")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return auc_train, auc_test


# Probar la función con el modelo baseline
auc_train_baseline, auc_test_baseline = calcular_metricas(
    baseline_model, X_train, y_train, X_test, y_test
)

## 4. Árbol de decisión

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

print("Árbol de decisión regularizado entrenado correctamente")
print(f"Número de nodos: {dt_model.tree_.node_count}")
print(f"Profundidad: {dt_model.tree_.max_depth}")

auc_train, auc_test = calcular_metricas(
    dt_model, X_train, y_train, X_test, y_test
)

In [ ]:

# aun más overfitting
dt_model2 = DecisionTreeClassifier(max_depth=None, min_samples_leaf=1, random_state=42)
dt_model2.fit(X_train, y_train)

print("Árbol de decisión regularizado entrenado correctamente")
print(f"Número de nodos: {dt_model2.tree_.node_count}")
print(f"Profundidad: {dt_model2.tree_.max_depth}")

auc_train, auc_test = calcular_metricas(
    dt_model2, X_train, y_train, X_test, y_test
)


In [ ]:
# menos overfitting
from sklearn.tree import DecisionTreeClassifier

dt_model3 = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42)
dt_model3.fit(X_train, y_train)

print("Árbol de decisión regularizado entrenado correctamente")
print(f"Número de nodos: {dt_model3.tree_.node_count}")
print(f"Profundidad: {dt_model3.tree_.max_depth}")

auc_train_regularized, auc_test_regularized = calcular_metricas(
    dt_model3, X_train, y_train, X_test, y_test
)

## 5. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10, min_samples_leaf=5)
rf_model.fit(X_train, y_train)

print("Random Forest Classifier entrenado correctamente")

# Probar la función con el modelo Random Forest
auc_train_rf, auc_test_rf = calcular_metricas(
    rf_model, X_train, y_train, X_test, y_test
)

In [ ]:
# Obtener la importancia de las características del modelo Random Forest
importances = rf_model.feature_importances_

# Crear una serie de pandas para asociar la importancia con el nombre de la característica
feature_importances = pd.Series(importances, index=X_train.columns)

# Ordenar las características por importancia en orden descendente
feature_importances = feature_importances.sort_values(ascending=False)

# Crear el gráfico de barras
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette='viridis')
plt.title('Importancia de las Características en el Modelo Random Forest')
plt.xlabel('Importancia')
plt.ylabel('Característica')
plt.tight_layout()
plt.show()

## 6. Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Instantiate and train the Gradient Boosting Classifier
# Parameters chosen for a good balance of performance and to prevent immediate overfitting
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

print("Gradient Boosting Classifier entrenado correctamente")

# Probar la función con el modelo Gradient Boosting
auc_train_gb, auc_test_gb = calcular_metricas(
    gb_model, X_train, y_train, X_test, y_test
)

## 7. Optimización de Hiperparámetros

In [ ]:
!pip install optuna
import optuna

In [ ]:
def objective(trial):
    """
    Define the objective function for Optuna to optimize.
    This function suggests hyperparameters, trains a GradientBoostingClassifier,
    and returns the ROC AUC score on the test set.
    """
    n_estimators = trial.suggest_int('n_estimators', 50, 300, step=50)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50)

    model = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    model.fit(X_train, y_train)

    # Calculate ROC AUC on the test set
    y_test_prob = model.predict_proba(X_test)[:, 1]
    auc_test = roc_auc_score(y_test, y_test_prob)

    return auc_test

print("Objective function 'objective' defined successfully.")

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Número de pruebas terminadas: {}".format(len(study.trials)))
print("Mejor prueba:")
trial = study.best_trial

print("  Valor (ROC AUC): {:.4f}".format(trial.value))
print("  Mejores hiperparámetros:")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))